# GLM-5.1 Cookbook

**Model:** `glm-5.1`  
**Latest update:** April 2026  
**Provider:** Z.AI

GLM-5.1 is Z.AI's latest flagship model, designed for long-horizon autonomous tasks. It can work continuously and autonomously on a single task for up to 8 hours, completing the full loop from planning and execution to iterative optimization and delivering production-grade results.

**Key specs from docs:**
- Input modalities: text
- Output modalities: text
- Context length: 200K tokens (industry-leading)
- Maximum output tokens: 128K
- Thinking enabled: Optional extended reasoning mode
- Benchmark performance: Aligned with Claude Opus 4.6 (58.4 on SWE-Bench Pro)
- Sustained execution: Up to 8 hours on single tasks

**Best-fit workloads:**
- Agentic coding and software engineering
- Long-horizon autonomous task execution
- Complex system optimization and engineering
- Iterative code generation with autonomous testing and refinement
- Multi-step reasoning and problem-solving
- MCP-integrated agent systems
- General conversation and reasoning
- Creative writing and artifact generation
- Office productivity and documentation

**Table of Contents:**
1. Setup and Installation
2. Basic Chat Completion
3. Thinking and Extended Reasoning
4. Tool Use and Function Calling
5. Streaming Responses
6. Long-Context and Multi-Turn Conversations
7. Long-Horizon Task Execution (Autonomous Agents)
8. Engineering and Optimization Tasks
9. Error Handling and Best Practices

## 1. Setup and Installation

In [ ]:
import subprocess
import sys

# Install required packages
install_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--disable-pip-version-check",
    "openai",  # Z.AI SDK is compatible with OpenAI SDK
    "requests",
]

result = subprocess.run(install_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Required packages installed successfully")
else:
    print("❌ Package installation failed")
    if result.stderr:
        print(result.stderr.strip())

In [ ]:
import os
from typing import Optional, List, Dict, Any
import json

# Configuration
# Set your API key here or use environment variable
API_KEY = os.getenv('ZAI_API_KEY', 'your-api-key-here')
API_BASE = "https://api.z.ai/api/paas/v4"
MODEL = "glm-5.1"

if API_KEY == 'your-api-key-here':
    print("⚠️  Warning: ZAI_API_KEY environment variable not set")
    print("Set it with: export ZAI_API_KEY='your-actual-api-key'")
else:
    print(f"✓ Model: {MODEL}")
    print(f"✓ API Key configured: {API_KEY[:20]}...")
    print(f"✓ API Base: {API_BASE}")

## 2. Basic Chat Completion

Start with a simple text-based conversation with GLM-5.1.

In [ ]:
import requests

def make_api_request(
    messages: List[Dict[str, str]],
    temperature: float = 0.7,
    max_tokens: int = 2048,
    stream: bool = False,
    thinking: Optional[Dict] = None,
    **kwargs
) -> Dict[str, Any]:
    """
    Make a request to GLM-5.1 API.
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        temperature: Sampling temperature (0.0 to 2.0)
        max_tokens: Maximum tokens in response
        stream: Whether to stream the response
        thinking: Optional thinking config {"type": "enabled"}
    
    Returns:
        API response dict
    """
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    }
    
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": stream,
    }
    
    if thinking:
        payload["thinking"] = thinking
    
    payload.update(kwargs)
    
    try:
        response = requests.post(
            f"{API_BASE}/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ API request failed: {e}")
        return {"error": str(e)}

print("✓ make_api_request() function defined")

In [ ]:
def simple_chat(user_message: str) -> str:
    """
    Simple single-turn conversation with GLM-5.1.
    """
    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]
    
    response = make_api_request(messages)
    
    if "error" in response:
        return f"Error: {response['error']}"
    
    try:
        return response["choices"][0]["message"]["content"]
    except (KeyError, IndexError) as e:
        return f"Failed to parse response: {e}"

# Example usage (uncomment to run)
# result = simple_chat("What are the key features of GLM-5.1?")
# print(result)

print("✓ simple_chat() function defined")

## 3. Thinking and Extended Reasoning

GLM-5.1 supports extended thinking mode for complex reasoning tasks. This allows the model to think through problems step-by-step before responding.

In [ ]:
def chat_with_thinking(user_message: str, max_tokens: int = 4096) -> Dict[str, Any]:
    """
    Chat with thinking enabled for complex reasoning.
    
    Thinking can help with:
    - Complex multi-step problems
    - Mathematical reasoning
    - Algorithm design
    - Code optimization decisions
    """
    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]
    
    response = make_api_request(
        messages,
        max_tokens=max_tokens,
        thinking={"type": "enabled"}
    )
    
    if "error" in response:
        return {"error": response["error"]}
    
    try:
        choice = response["choices"][0]["message"]
        return {
            "thinking": choice.get("thought", ""),
            "response": choice.get("content", ""),
            "usage": response.get("usage", {})
        }
    except (KeyError, IndexError) as e:
        return {"error": f"Failed to parse response: {e}"}

# Example usage (uncomment to run)
# result = chat_with_thinking(
#     "Design an efficient algorithm to find the longest increasing subsequence in O(n log n) time"
# )
# if "error" not in result:
#     print("Thinking process:")
#     print(result["thinking"][:500] + "...")
#     print("\nFinal response:")
#     print(result["response"])

print("✓ chat_with_thinking() function defined")

## 4. Tool Use and Function Calling

GLM-5.1 supports tool use and function calling through the chat API.

In [ ]:
def chat_with_tools(
    user_message: str,
    tools: List[Dict[str, Any]],
    tool_choice: str = "auto"
) -> Dict[str, Any]:
    """
    Chat with tool/function calling enabled.
    
    Args:
        user_message: The user's request
        tools: List of tool definitions
        tool_choice: "auto", "required", or tool name
    
    Returns:
        Response with potential tool calls
    """
    messages = [{"role": "user", "content": user_message}]
    
    response = make_api_request(
        messages,
        tools=tools,
        tool_choice=tool_choice
    )
    
    if "error" in response:
        return {"error": response["error"]}
    
    try:
        choice = response["choices"][0]
        return {
            "content": choice["message"].get("content", ""),
            "tool_calls": choice["message"].get("tool_calls", []),
            "stop_reason": choice.get("finish_reason", "")
        }
    except (KeyError, IndexError) as e:
        return {"error": f"Failed to parse response: {e}"}

# Example tool definition
EXAMPLE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name or coordinates"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_database",
            "description": "Search a database for records",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    },
                    "filters": {
                        "type": "object",
                        "description": "Optional filters"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("✓ chat_with_tools() function defined")
print(f"✓ {len(EXAMPLE_TOOLS)} example tools defined")

## 5. Streaming Responses

For real-time applications, stream responses as they're generated.

In [ ]:
def stream_response(user_message: str) -> None:
    """
    Stream response from GLM-5.1 token-by-token.
    
    Useful for:
    - Real-time user interfaces
    - Immediate feedback during generation
    - Progressive output processing
    """
    messages = [{"role": "user", "content": user_message}]
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    }
    
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": True,
        "temperature": 0.7,
        "max_tokens": 2048
    }
    
    try:
        response = requests.post(
            f"{API_BASE}/chat/completions",
            headers=headers,
            json=payload,
            stream=True,
            timeout=30
        )
        response.raise_for_status()
        
        print("📝 Response: ", end="", flush=True)
        for line in response.iter_lines():
            if not line:
                continue
            if line.startswith(b"data: "):
                try:
                    data = json.loads(line[6:])
                    if "choices" in data and len(data["choices"]) > 0:
                        delta = data["choices"][0].get("delta", {})
                        if "content" in delta:
                            print(delta["content"], end="", flush=True)
                except json.JSONDecodeError:
                    pass
        print("\n")
    except requests.exceptions.RequestException as e:
        print(f"❌ Streaming failed: {e}")

# Example usage (uncomment to run)
# stream_response("Explain the concept of neural networks in 3 sentences")

print("✓ stream_response() function defined")

## 6. Long-Context and Multi-Turn Conversations

GLM-5.1 supports 200K tokens of context, enabling long documents and multi-turn conversations.

In [ ]:
class ConversationManager:
    """
    Manages multi-turn conversations with GLM-5.1.
    
    Example:
        conv = ConversationManager()
        conv.add_user_message("What is machine learning?")
        response = conv.get_response()
    """
    
    def __init__(self, system_prompt: str = ""):
        self.messages = []
        if system_prompt:
            self.system_prompt = system_prompt
        else:
            self.system_prompt = "You are a helpful AI assistant."
    
    def add_user_message(self, content: str) -> None:
        """Add a user message to the conversation."""
        self.messages.append({"role": "user", "content": content})
    
    def add_assistant_message(self, content: str) -> None:
        """Add an assistant message to the conversation."""
        self.messages.append({"role": "assistant", "content": content})
    
    def get_response(self, temperature: float = 0.7, max_tokens: int = 2048) -> str:
        """Get a response from GLM-5.1 based on conversation history."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            *self.messages
        ]
        
        response = make_api_request(messages, temperature=temperature, max_tokens=max_tokens)
        
        if "error" in response:
            return f"Error: {response['error']}"
        
        try:
            content = response["choices"][0]["message"]["content"]
            self.add_assistant_message(content)
            return content
        except (KeyError, IndexError) as e:
            return f"Failed to parse response: {e}"
    
    def clear_history(self) -> None:
        """Clear conversation history."""
        self.messages = []
    
    def get_conversation(self) -> List[Dict[str, str]]:
        """Get the full conversation history."""
        return self.messages.copy()

# Example usage
conv = ConversationManager(
    system_prompt="You are an expert Python programmer."
)

# Multi-turn conversation simulation
# conv.add_user_message("What's a decorator in Python?")
# response1 = conv.get_response()
# print("Assistant:", response1)
#
# conv.add_user_message("Can you show me an example?")
# response2 = conv.get_response()
# print("Assistant:", response2)

print("✓ ConversationManager class defined")

## 7. Long-Horizon Task Execution (Autonomous Agents)

GLM-5.1 can work autonomously on single tasks for up to 8 hours. Here's a pattern for implementing agentic behavior.

In [ ]:
from datetime import datetime, timedelta
from enum import Enum

class TaskStatus(Enum):
    STARTED = "started"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"

class AutomatedTask:
    """
    Represents a long-horizon automated task executed by GLM-5.1.
    
    GLM-5.1's 8-hour sustained execution capability enables:
    - Planning and execution loops
    - Iterative optimization
    - Continuous testing and refinement
    - Complex engineering tasks
    """
    
    def __init__(
        self,
        task_id: str,
        objective: str,
        max_duration_hours: int = 8,
        max_iterations: int = 100
    ):
        self.task_id = task_id
        self.objective = objective
        self.max_duration_hours = max_duration_hours
        self.max_iterations = max_iterations
        
        self.status = TaskStatus.STARTED
        self.start_time = datetime.now()
        self.end_time: Optional[datetime] = None
        self.iterations = 0
        self.execution_log = []
        self.current_result = ""
    
    def is_expired(self) -> bool:
        """Check if task has exceeded maximum duration."""
        elapsed = datetime.now() - self.start_time
        return elapsed > timedelta(hours=self.max_duration_hours)
    
    def can_continue(self) -> bool:
        """Check if task can continue iterating."""
        return (
            self.status == TaskStatus.IN_PROGRESS and
            not self.is_expired() and
            self.iterations < self.max_iterations
        )
    
    def log_step(self, step_description: str, result: str) -> None:
        """Log a single iteration/step."""
        self.execution_log.append({
            "iteration": self.iterations,
            "timestamp": datetime.now().isoformat(),
            "description": step_description,
            "result": result
        })
        self.current_result = result
        self.iterations += 1
    
    def complete(self, success: bool = True) -> None:
        """Mark task as completed."""
        self.end_time = datetime.now()
        self.status = TaskStatus.COMPLETED if success else TaskStatus.FAILED
    
    def get_duration(self) -> timedelta:
        """Get total task duration."""
        end = self.end_time or datetime.now()
        return end - self.start_time
    
    def get_summary(self) -> Dict[str, Any]:
        """Get task summary."""
        return {
            "task_id": self.task_id,
            "objective": self.objective,
            "status": self.status.value,
            "iterations": self.iterations,
            "duration": str(self.get_duration()),
            "final_result": self.current_result
        }

# Example: Setting up an autonomous coding task
coding_task = AutomatedTask(
    task_id="task-001",
    objective="Optimize database query performance and achieve 3x speedup",
    max_duration_hours=8,
    max_iterations=100
)

# This would be executed in a loop:
# coding_task.status = TaskStatus.IN_PROGRESS
# while coding_task.can_continue():
#     # GLM-5.1 agents would:
#     # 1. Analyze current performance
#     # 2. Identify bottlenecks
#     # 3. Propose optimization
#     # 4. Implement and test
#     # 5. Measure improvement
#     coding_task.log_step("iteration_description", "result")
#
# coding_task.complete(success=True)
# print(coding_task.get_summary())

print("✓ AutomatedTask class defined")

## 8. Engineering and Optimization Tasks

GLM-5.1 excels at iterative optimization and engineering tasks. Here's a pattern for code optimization.

In [ ]:
class CodeOptimizer:
    """
    Represents GLM-5.1's autonomous code optimization capability.
    
    Based on real benchmarks:
    - 58.4 on SWE-Bench Pro (beating GPT-5.4, Claude Opus 4.6)
    - 3.6x geometric mean speedup on KernelBench Level 3
    - Thousands of tool-invocation-driven optimizations
    """
    
    def __init__(
        self,
        original_code: str,
        performance_metric: str = "execution_time"
    ):
        self.original_code = original_code
        self.performance_metric = performance_metric
        self.optimization_history = []
        self.current_code = original_code
        self.current_performance = None
    
    def request_optimization_suggestion(self, bottleneck_analysis: str) -> Dict[str, str]:
        """
        Ask GLM-5.1 to suggest optimizations based on performance analysis.
        
        The model can:
        - Identify algorithmic improvements
        - Suggest data structure changes
        - Recommend caching strategies
        - Propose parallelization
        """
        prompt = f"""
        Current code has the following performance bottleneck:
        {bottleneck_analysis}
        
        Analyze the code and suggest specific optimizations:
        
        ```python
        {self.current_code}
        ```
        
        Provide:
        1. Root cause analysis
        2. Specific optimization strategy
        3. Optimized code
        4. Expected performance improvement
        """
        
        # This would call GLM-5.1 with thinking enabled
        # response = chat_with_thinking(prompt)
        # return {"suggestion": response["response"]}
        
        return {"suggestion": "Optimization suggestion from GLM-5.1"}
    
    def apply_optimization(self, optimized_code: str, improvement_factor: float) -> None:
        """
        Apply an optimization and track it.
        """
        self.optimization_history.append({
            "iteration": len(self.optimization_history) + 1,
            "previous_code": self.current_code,
            "improvement_factor": improvement_factor,
            "timestamp": datetime.now().isoformat()
        })
        self.current_code = optimized_code
        self.current_performance = improvement_factor
    
    def get_optimization_summary(self) -> Dict[str, Any]:
        """
        Get summary of all optimizations applied.
        """
        total_improvement = 1.0
        for opt in self.optimization_history:
            total_improvement *= opt["improvement_factor"]
        
        return {
            "total_iterations": len(self.optimization_history),
            "cumulative_improvement": total_improvement,
            "optimization_history": self.optimization_history
        }

print("✓ CodeOptimizer class defined")

## 9. Error Handling and Best Practices

In [ ]:
class GLM51Client:
    """
    Production-ready client for GLM-5.1 with comprehensive error handling.
    
    Best practices implemented:
    - Automatic retries with exponential backoff
    - Rate limiting awareness
    - Timeout management
    - Response validation
    - Cost tracking (token usage)
    """
    
    def __init__(
        self,
        api_key: str,
        api_base: str = "https://api.z.ai/api/paas/v4",
        model: str = "glm-5.1",
        max_retries: int = 3
    ):
        self.api_key = api_key
        self.api_base = api_base
        self.model = model
        self.max_retries = max_retries
        self.total_tokens_used = 0
        self.request_count = 0
    
    def _get_headers(self) -> Dict[str, str]:
        """Get request headers."""
        return {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}"
        }
    
    def make_request(
        self,
        messages: List[Dict[str, str]],
        **kwargs
    ) -> Dict[str, Any]:
        """
        Make API request with error handling and retries.
        """
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": kwargs.get("temperature", 0.7),
            "max_tokens": kwargs.get("max_tokens", 2048),
        }
        payload.update({k: v for k, v in kwargs.items() if k not in ["temperature", "max_tokens"]})
        
        for attempt in range(self.max_retries):
            try:
                response = requests.post(
                    f"{self.api_base}/chat/completions",
                    headers=self._get_headers(),
                    json=payload,
                    timeout=60
                )
                response.raise_for_status()
                
                data = response.json()
                self.request_count += 1
                
                # Track token usage
                if "usage" in data:
                    self.total_tokens_used += data["usage"].get("total_tokens", 0)
                
                return data
            
            except requests.exceptions.Timeout:
                if attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⏱️  Timeout. Retrying in {wait_time}s...")
                    import time
                    time.sleep(wait_time)
                else:
                    return {"error": "Request timeout after retries"}
            
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    if attempt < self.max_retries - 1:
                        wait_time = 2 ** attempt
                        print(f"⚠️  Rate limited. Retrying in {wait_time}s...")
                        import time
                        time.sleep(wait_time)
                    else:
                        return {"error": "Rate limit exceeded"}
                else:
                    return {"error": f"HTTP {e.response.status_code}: {e.response.text}"}
            
            except Exception as e:
                return {"error": f"Unexpected error: {str(e)}"}
        
        return {"error": "Max retries exceeded"}
    
    def get_stats(self) -> Dict[str, Any]:
        """Get client usage statistics."""
        return {
            "requests_made": self.request_count,
            "total_tokens_used": self.total_tokens_used,
            "avg_tokens_per_request": (
                self.total_tokens_used / self.request_count
                if self.request_count > 0 else 0
            )
        }

print("✓ GLM51Client class defined")

### Best Practices Summary

1. **API Key Management**
   - Always use environment variables for API keys
   - Never hardcode credentials in notebook cells
   - Rotate keys regularly

2. **Token Usage**
   - Monitor token consumption (GLM-5.1 has 200K context)
   - Use `max_tokens` to control response length
   - Implement cost tracking for production systems

3. **Long-Horizon Tasks**
   - GLM-5.1 can work for up to 8 hours on single tasks
   - Implement checkpointing for fault tolerance
   - Log execution steps for debugging

4. **Tool Use and MCP**
   - Define tools clearly with type hints
   - Validate tool calls before execution
   - Implement tool result feedback loops

5. **Error Handling**
   - Implement exponential backoff for retries
   - Handle rate limits (429 status)
   - Validate response structure before parsing

6. **Performance Optimization**
   - Use streaming for real-time applications
   - Leverage thinking mode for complex reasoning
   - Batch requests when possible

7. **Testing and Validation**
   - Test with small models before scaling
   - Validate outputs match expected format
   - Monitor model accuracy metrics in production

## Resources and References

- **Official Documentation:** https://docs.z.ai/guides/llm/glm-5.1
- **API Reference:** https://docs.z.ai/api-reference/llm/chat-completion
- **Migration Guide:** https://docs.z.ai/guides/overview/migrate-to-glm-new
- **Community:** [Discord](https://discord.gg/QR7SARHRxK), [GitHub](https://github.com/zai-org), [LinkedIn](https://www.linkedin.com/company/zdotai)

## Key Capabilities Summary

| Feature | Capability |
|---------|-------------|
| Context Length | 200K tokens |
| Max Output | 128K tokens |
| Sustained Execution | 8 hours on single tasks |
| Coding Performance | 58.4 on SWE-Bench Pro |
| General Intelligence | Aligned with Claude Opus 4.6 |
| Input Modalities | Text |
| Output Modalities | Text |
| Extended Thinking | Yes (optional reasoning) |
| Tool/Function Use | Yes (MCP compatible) |
| Streaming | Yes |

---

**Last Updated:** April 2026  
**Model Version:** GLM-5.1  
**Status:** Production Ready